# 🌡️ Engenharia de Features: Índice de Risco Fiscal (IRF v2)
## Fase 3: Data Preparation (Modeling Features)
**Autor:** Luiz Maibashi (AI Scientist/Engineer)

---

### 📋 Business Understanding
**🤔 POR QUÊ:** 
Um motor de compliance AML que olha apenas para o valor da transação é limitado e gera muitos falsos positivos. Para sermos sêniores, precisamos de uma **âncora contextual**.

O IRF v2 é um termômetro que resume 6 sinais macroeconômicos em uma única métrica de 0 a 100. 
**Objetivo:** Criar uma feature que informe ao modelo de ML o nível de stress do mercado brasileiro no exato momento da transação. Isso permite distinguir entre a "fuga para segurança" (legítima) e o "smurfing" (criminoso).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import sys
from pathlib import Path

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [12, 6]

sys.path.append(str(Path.cwd().parent / "src"))
from utils import carregar_dataset_mestre, calcular_irf_v2, DATA_PROC

df = carregar_dataset_mestre()
print("✅ Dados macroeconômicos integrados para construção do índice.")

### 🔍 Análise de Multicolinearidade (VIF)
**🤔 POR QUÊ:** Se duas variáveis medem a mesma coisa (ex: Dólar e Dólar Ajustado), o índice ficará "viciado". Precisamos garantir que os sinais sejam o mais ortogonais possível.

**⚙️ COMO:** Vamos olhar a matriz de correlação das features candidatas.

In [ ]:
cols_irf = ['divida_bruta_pib', 'brl_usd_var_30d', 'desvio_meta_ipca', 'selic_meta', 'usdt_volume_mm30', 'ibc_br']
df_stats = df[cols_irf].dropna()

corr = df_stats.corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Matriz de Correlação das Variáveis Macro")
plt.show()

# 📊 O QUE DIZ: Alta correlação entre IPCA e Selic é esperada. 
# Mas a Dívida/PIB costuma ser o sinal mais isolado e poderoso de risco fiscal.

### 🧬 Justificativa Científica dos Pesos (PCA)
**🤔 POR QUÊ:** Em vez de chutar que o Câmbio vale 40%, vamos deixar os dados falarem. A **Análise de Componentes Principais (PCA)** nos diz qual variável explica a maior parte da variância (o "balanço") do sistema macro.

In [ ]:
X = StandardScaler().fit_transform(df_stats)
pca = PCA(n_components=1)
pca.fit(X)

loadings = pd.DataFrame(
    pca.components_.T, 
    columns=['Componente_Risco'], 
    index=cols_irf
)
display(loadings.sort_values(by='Componente_Risco', ascending=False))

# 📊 O QUE DIZ: O PCA nos ajuda a calibrar os pesos do calcular_irf_v2(). 
# As variáveis com maiores 'loadings' são as âncoras do risco fiscal real.

### 🏗️ Cálculo do IRF v2 (Produção)
**🤔 POR QUÊ:** Agora consolidamos os sinais em uma escala interpretável (0-100).

In [ ]:
# ⚙️ COMO: Aplicando a função unificada do src/utils.py para garantir paridade
df['irf_v2'] = df.apply(lambda row: calcular_irf_v2(
    score_tom_copom=0.5, # Placeholder se não houver ata no dia
    brl_adj_dxy_30d=row.get('brl_adj_dxy_30d', 0),
    ipca_desvio_meta=row.get('desvio_meta_ipca', 0),
    variacao_usdt_30d=row.get('usdt_volume_mm30', 0),
    divida_pib_var=row.get('divida_pib_var', 0),
    ibc_br_var=row.get('ibc_br', 0)
), axis=1)

df['irf_v2_mm7d'] = df['irf_v2'].rolling(7).mean()
print("✅ Índice de Risco Fiscal v2 calculado.")

### 📈 Validação Histórica
**🤔 POR QUÊ:** Um índice que não reflete a realidade não serve para compliance. Vamos checar se o IRF subiu em momentos críticos.

In [ ]:
plt.plot(df.index, df['irf_v2'], alpha=0.3, label='Diário')
plt.plot(df.index, df['irf_v2_mm7d'], color='red', linewidth=2, label='Tendência (MM7D)')
plt.axhline(70, color='darkred', linestyle='--', label='Stress Crítico')
plt.axhline(40, color='orange', linestyle='--', label='Atenção')
plt.title("Evolução do Índice de Risco Fiscal Brasil (IRF v2)")
plt.legend()
plt.show()

# 📊 O QUE DIZ: Se o índice ultrapassa 70, o modelo de ML deve ser mais tolerante 
# a grandes volumes de USDT, pois o cenário macro explica o comportamento.

### 💾 Persistência dos Dados de Inteligência
**🤔 POR QUÊ:** Salvar o dataset mestre processado para que o motor de compliance possa consumi-lo via API ou Batch.

In [ ]:
output_path = DATA_PROC / "dataset_irf_completo.csv"
df.to_csv(output_path)
print(f"💾 Dataset de Inteligência salvo em: {output_path}")

## 🏁 Conclusão da Fase de Data Preparation

**O que fizemos:**
1. **Engenharia de Features:** Criamos o IRF v2, um indicador sintético que resume o cenário macro brasileiro.
2. **Rigor de Pesos:** Usamos PCA para garantir que o índice reflita a variância real dos dados, e não suposições arbitrárias.
3. **Dataset de Inteligência:** Consolidamos todas as variáveis em um único dataset processado e pronto para modelagem.

**Próximos Passos:**
Com o IRF v2 pronto, seguiremos para o **Notebook 03**, onde treinaremos o **Motor de Anomalias (Isolation Forest)** para detectar comportamentos suspeitos cruzando dados de transação com este índice fiscal.